In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import GradientBoostingRegressor

import os


In [8]:
print(os.getcwd())

/Users/bryancruz/Documents/UCSD/6th Quarter/DSC 148/Wave Power Project


In [9]:
# Load one dataset first
df = pd.read_csv("../data/WEC/WEC_Sydney_49.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/WEC/WEC_Sydney_49.csv'

In [ ]:
# Check shape and columns
print(df.shape)
print(df.columns)

In [ ]:
y = df["Total_Power"]

Did not include Total_Power, and I left out the individual Power columns at first because they may make the task too easy/leaky

In [ ]:
# Use only X and Y coordinate columns
coord_cols = [col for col in df.columns if col.startswith("X") or col.startswith("Y")]

X = df[coord_cols]
y = df["Total_Power"]

print(X.shape)
print(y.shape)

In [ ]:
# First split off the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

# Then split remaining data into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,  # gives ~15% overall
    random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)
    
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    
    results = {
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_preds),
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_preds)),
        "Val RMSE": np.sqrt(mean_squared_error(y_val, val_preds)),
        "Train R2": r2_score(y_train, train_preds),
        "Val R2": r2_score(y_val, val_preds)
    }
    
    return results

In [ ]:
dummy = DummyRegressor(strategy="mean")

dummy_results = evaluate_model(
    dummy,
    X_train, y_train,
    X_val, y_val,
    "Dummy Baseline"
)

dummy_results

In [ ]:
linear = LinearRegression()

linear_results = evaluate_model(
    linear,
    X_train, y_train,
    X_val, y_val,
    "Linear Regression"
)

linear_results

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    rf,
    X_train, y_train,
    X_val, y_val,
    "Random Forest"
)

rf_results

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_results = evaluate_model(
    gb,
    X_train, y_train,
    X_val, y_val,
    "Gradient Boosting"
)

gb_results

In [ ]:
hgb = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hgb_results = evaluate_model(
    hgb,
    X_train, y_train,
    X_val, y_val,
    "Hist Gradient Boosting"
)

hgb_results

In [ ]:
results_df = pd.DataFrame([
    dummy_results,
    linear_results,
    rf_results,
    gb_results,
    hgb_results
])

results_df.sort_values("Val MAE")

In [ ]:
best_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

# Combine train + validation
X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])

# Retrain on all available training data
best_model.fit(X_full_train, y_full_train)

# Final test predictions
test_preds = best_model.predict(X_test)

print("Final Test MAE:",
      mean_absolute_error(y_test, test_preds))

print("Final Test RMSE:",
      np.sqrt(mean_squared_error(y_test, test_preds)))

print("Final Test R2:",
      r2_score(y_test, test_preds))

Hist Gradient Boosting performed best overall, with the lowest validation MAE and highest validation R². 
This suggests that nonlinear boosted trees captured the relationship between WEC layout coordinates and Total_Power better than Linear Regression, Random Forest, and standard Gradient Boosting.

In [ ]:
# raw coordinate features only
X_raw = df[coord_cols]

X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42
)

rf_raw = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_raw_results = evaluate_model(
    rf_raw,
    X_train, y_train,
    X_val, y_val,
    "RF Raw Coordinates"
)

hgb_raw = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hgb_raw_results = evaluate_model(
    hgb_raw,
    X_train, y_train,
    X_val, y_val,
    "HGB Raw Coordinates"
)

NameError: name 'df' is not defined

### Feature engineering

In [ ]:
def add_geometry_features(df, n_devices=49):

    df = df.copy()

    x_cols = [f"X{i}" for i in range(1, n_devices + 1)]
    y_cols = [f"Y{i}" for i in range(1, n_devices + 1)]

    X_vals = df[x_cols].values
    Y_vals = df[y_cols].values

    # centroid
    df["centroid_x"] = X_vals.mean(axis=1)
    df["centroid_y"] = Y_vals.mean(axis=1)

    # spread
    df["x_std"] = X_vals.std(axis=1)
    df["y_std"] = Y_vals.std(axis=1)

    # range
    df["x_range"] = X_vals.max(axis=1) - X_vals.min(axis=1)
    df["y_range"] = Y_vals.max(axis=1) - Y_vals.min(axis=1)

    # distance from center
    cx = X_vals.mean(axis=1).reshape(-1, 1)
    cy = Y_vals.mean(axis=1).reshape(-1, 1)

    distances = np.sqrt((X_vals - cx)**2 + (Y_vals - cy)**2)

    df["mean_dist_center"] = distances.mean(axis=1)
    df["max_dist_center"] = distances.max(axis=1)

    return df

In [ ]:
df_fe = add_geometry_features(df, n_devices=49)

coord_cols = [
    col for col in df_fe.columns
    if col.startswith("X") or col.startswith("Y")
]

geometry_cols = [
    "centroid_x",
    "centroid_y",
    "x_std",
    "y_std",
    "x_range",
    "y_range",
    "mean_dist_center",
    "max_dist_center"
]

X = df_fe[coord_cols + geometry_cols]
y = df_fe["Total_Power"]

In [ ]:
# split again after rebuilding X
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42
)

# retrain random forest on the new X
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    rf,
    X_train, y_train,
    X_val, y_val,
    "Random Forest with Geometry Features"
)

rf_results

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values("Importance", ascending=False)

importance_df.head(15)

In [ ]:
import matplotlib.pyplot as plt

top_features = importance_df.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top Random Forest Feature Importances")
plt.show()

The model is mostly using layout-spread features, not individual device coordinates. That means Total_Power is strongly related to how spread out the farm layout is, especially in the x-direction. The Random Forest feature importance results show that engineered geometry features dominate the prediction. The most important variables were x_std and mean_dist_center, each contributing over 43% of the model’s total importance. This suggests that overall farm spread is more predictive of total power than any single WEC’s raw coordinate. In particular, horizontal spread appears especially important because x_std and x_range rank above their y-direction equivalents.

Random Forest importance can favor correlated features. x_std and mean_dist_center are probably related

In [ ]:
df_fe[["x_std", "mean_dist_center", "x_range", "y_std"]].corr()

Correlation analysis showed that x_std and mean_dist_center were highly correlated (0.947), indicating that both features capture similar geometric properties of the WEC farm layout. This suggests the model relies heavily on overall spatial dispersion of devices rather than isolated coordinates.

In [ ]:
top_geom = df_fe[
    ["x_std", "mean_dist_center",
     "x_range", "y_std", "Total_Power"]
]

plt.figure(figsize=(8,6))

sns.heatmap(
    top_geom.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap of Key Geometry Features")
plt.show()

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "RF Raw Coordinates",
        "Val MAE": rf_raw_results["Val MAE"],
        "Val RMSE": rf_raw_results["Val RMSE"],
        "Val R2": rf_raw_results["Val R2"]
    },

    {
        "Model": "RF + Geometry Features",
        "Val MAE": rf_results["Val MAE"],
        "Val RMSE": rf_results["Val RMSE"],
        "Val R2": rf_results["Val R2"]
    },

    {
        "Model": "HGB Raw Coordinates",
        "Val MAE": hgb_raw_results["Val MAE"],
        "Val RMSE": hgb_raw_results["Val RMSE"],
        "Val R2": hgb_raw_results["Val R2"]
    },

    {
        "Model": "HGB + Geometry Features",
        "Val MAE": hgb_results["Val MAE"],
        "Val RMSE": hgb_results["Val RMSE"],
        "Val R2": hgb_results["Val R2"]
    }
])

comparison_df.sort_values("Val MAE")

Adding engineered geometry features substantially improved Random Forest performance, reducing validation MAE from approximately 3669 to 2731. This suggests that geometric summaries provide a more informative representation of WEC farm layouts than raw coordinates alone.

HistGradientBoosting performance remained nearly unchanged after feature engineering, suggesting the boosting model was already capable of extracting geometric structure from the raw coordinate inputs.

In [ ]:
rf_mae_pct = (
    rf_results["Val MAE"] / y.mean()
) * 100

print(rf_mae_pct)

NameError: name 'rf_results' is not defined

In [ ]:
print(f"Percent Error: {rf_mae_pct:.4f}%")

In [ ]:
# Make sure X includes raw coordinates + geometry features
X = df_fe[coord_cols + geometry_cols]
y = df_fe["Total_Power"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42
)

# Retrain RF on geometry-feature data
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

best_preds = rf.predict(X_test)

In [ ]:
best_preds = rf.predict(X_test)

plt.figure(figsize=(7,7))

plt.scatter(y_test, best_preds, alpha=0.3)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel("Actual Total Power")
plt.ylabel("Predicted Total Power")
plt.title("Predicted vs Actual Total Power")

plt.show()

The predicted-vs-actual plot shows that the Random Forest model predicts Total_Power with high accuracy across the full range of observed values. Most predictions lie close to the ideal diagonal line, indicating strong agreement between predicted and true power output.

residuals = y_test - best_preds

plt.figure(figsize=(8,5))

plt.scatter(best_preds, residuals, alpha=0.3)

plt.axhline(0, linestyle='--')

plt.xlabel("Predicted Total Power")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()

The residual plot shows that prediction errors remain centered around zero with no strong systematic bias. However, residual variance decreases for higher predicted power values, suggesting that highly efficient WEC layouts may follow more consistent geometric patterns that are easier for the model to learn.

In [ ]:
sydney_49 = pd.read_csv("../data/WEC/WEC_Sydney_49.csv")
sydney_100 = pd.read_csv("../data/WEC/WEC_Sydney_100.csv")
perth_49 = pd.read_csv("../data/WEC/WEC_Perth_49.csv")
perth_100 = pd.read_csv("../data/WEC/WEC_Perth_100.csv")

df_all = pd.concat([sydney_49, sydney_100, perth_49, perth_100], ignore_index=True)

In [ ]:
def run_rf_pipeline(df, n_devices, label):

    df = add_geometry_features(df, n_devices)

    coord_cols = [
        col for col in df.columns
        if col.startswith("X") or col.startswith("Y")
    ]

    geometry_cols = [
        "centroid_x",
        "centroid_y",
        "x_std",
        "y_std",
        "x_range",
        "y_range",
        "mean_dist_center",
        "max_dist_center"
    ]

    X = df[coord_cols + geometry_cols]
    y = df["Total_Power"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)

    results = {
    "Dataset": label,
    "MAE": mean_absolute_error(y_test, preds),
    "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
    "R2": r2_score(y_test, preds),
    "Percent_Error": (
        mean_absolute_error(y_test, preds) / y.mean()
    ) * 100
}

    return results

In [ ]:
comparison_results = []

comparison_results.append(
    run_rf_pipeline(perth_49, 49, "Perth 49")
)

comparison_results.append(
    run_rf_pipeline(sydney_49, 49, "Sydney 49")
)

comparison_results.append(
    run_rf_pipeline(perth_100, 100, "Perth 100")
)

comparison_results.append(
    run_rf_pipeline(sydney_100, 100, "Sydney 100")
)

comparison_df = pd.DataFrame(comparison_results)

comparison_df

In [ ]:
scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

scores = -scores

print("CV MAE Scores:", scores)
print("Mean CV MAE:", scores.mean())
print("Std CV MAE:", scores.std())

Five-fold cross-validation produced a mean MAE of 2363.89 with a standard deviation of only 47.06. The low variation across folds indicates that model performance is highly stable and does not depend heavily on a particular train-validation split. This suggests strong generalization ability and confirms that the Random Forest model is not simply memorizing the training data.

In [ ]:
results_df["MAE Gap"] = (
    results_df["Val MAE"] -
    results_df["Train MAE"]
)

results_df.sort_values("MAE Gap")

Random Forest achieved the best predictive accuracy but also exhibited the largest train-validation MAE gap, indicating some degree of overfitting. HistGradientBoosting showed a smaller gap and therefore stronger generalization behavior. However, Random Forest still maintained superior validation performance, making it the preferred surrogate model.

In [ ]:
perth_49.columns.tolist()

NameError: name 'perth_49' is not defined

While Total Power can be predicted very accurately from geometric layout, interaction efficiency represented by qW is more difficult to model because it captures complex hydrodynamic interactions among converters.

In [ ]:
print(len(y_val))
print(len(residuals))

In [ ]:
val_preds = rf.predict(X_val)

val_residuals = y_val - val_preds

plt.figure(figsize=(8,5))

plt.scatter(y_val, val_residuals, alpha=0.3)

plt.axhline(0, linestyle='--')

plt.xlabel("Actual Total Power")
plt.ylabel("Residuals")
plt.title("Validation Residual Plot")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(y_test, residuals, alpha=0.3)

plt.axhline(0, linestyle='--')

plt.xlabel("Actual Total Power")
plt.ylabel("Residuals")
plt.title("Test Residual Plot")

plt.show()

Residuals are centered around zero for both validation and test datasets, indicating little systematic prediction bias. The residual plots also show similar patterns across validation and test sets, supporting the model's ability to generalize. Prediction errors decrease for higher-power layouts, suggesting that highly efficient farm configurations exhibit more consistent geometric characteristics that are easier for the model to capture.

In [ ]:
print(type(best_preds))
print(best_preds.shape)

print(type(y_test))
print(y_test.shape)

In [ ]:
print(residuals[:10])

In [ ]:
df["qW"].describe()

In [ ]:
df["qW"].hist(bins=30)
plt.show()

In [ ]:
df_fe = add_geometry_features(df, n_devices=49)

coord_cols = [
    col for col in df_fe.columns
    if col.startswith("X") or col.startswith("Y")
]

geometry_cols = [
    "centroid_x",
    "centroid_y",
    "x_std",
    "y_std",
    "x_range",
    "y_range",
    "mean_dist_center",
    "max_dist_center"
]

X = df_fe[coord_cols + geometry_cols]
y = df_fe["qW"]

# split again after rebuilding X
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42
)

# retrain random forest on the new X
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    rf,
    X_train, y_train,
    X_val, y_val,
    "Random Forest with Geometry Features"
)

rf_results

Farm interaction efficiency (qW) is largely determined by the spatial arrangement of devices. The high validation R² indicates that geometric layout characteristics capture most of the variation in interaction efficiency. So The way you place the devices matters a lot for both total power production and interaction efficiency.

In [ ]:
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importances.sort_values(
    "Importance",
    ascending=False
).head(15)

For both targets, Total Power and qW (interaction efficiency) the model is relying on the same geometric properties. In fact, x_std + mean_dist_center account for 0.445 + 0.397 = 0.842 or about 84% of all feature importance. So then how spread out the devices are matters much more than the exact position of any individual device.

Feature importance analysis for both Total_Power and qW revealed remarkably similar patterns. In both cases, the most influential variables were x_std and mean_dist_center, together accounting for more than 80% of the model's total importance. Individual device coordinates contributed very little predictive power. This indicates that overall farm geometry, particularly spatial dispersion and average spacing between devices, is far more important than the precise location of any single converter. The consistency of these results across both targets suggests that geometric layout governs not only total energy production but also interaction efficiency within the farm.

In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X)

print("Original features:", X.shape[1])
print("PCA features:", X_pca.shape[1])
print("Explained variance:", pca.explained_variance_ratio_.sum())

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_pca,
    y,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42
)

rf_pca = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

pca_results = evaluate_model(
    rf_pca,
    X_train,
    y_train,
    X_val,
    y_val,
    "RF + PCA"
)

pd.DataFrame([pca_results])

In [ ]:
comparison = pd.DataFrame([
    rf_results,
    pca_results
])

comparison

PCA reduced the feature space from 106 variables to only 13 principal components while preserving approximately 95.8% of the variance. However, model performance decreased substantially, with validation R² dropping from 0.982 to 0.940. This suggests that although the farm layouts can be represented in a lower-dimensional space, the engineered geometric features contain physically meaningful information that is partially lost through dimensionality reduction.

The best-performing surrogate model was a Random Forest trained on both raw coordinates and engineered geometry features to predict Total_Power. The model achieved approximately 99% explained variance and generalized well under cross-validation. Additional experiments showed that the same geometric features also predict qW with high accuracy, indicating that farm-wide spatial dispersion is a key driver of both energy production and interaction efficiency.

In [ ]:
df_fe = add_geometry_features(df, n_devices=49)

coord_cols = [
    col for col in df_fe.columns
    if col.startswith("X") or col.startswith("Y")
]

geometry_cols = [
    "centroid_x", "centroid_y",
    "x_std", "y_std",
    "x_range", "y_range",
    "mean_dist_center", "max_dist_center"
]

X = df_fe[coord_cols + geometry_cols]
y = df_fe["Total_Power"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42
)

X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])

rf_final = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=1
)

rf_final.fit(X_full_train, y_full_train)

test_preds = rf_final.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, test_preds))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_preds)))
print("Test R2:", r2_score(y_test, test_preds))